# Week 1 Assessment — Enhanced Career Chatbot

**Author: Jemine Mene-Ejegi**

This notebook is the Week 1 exercise submission for `4_lab4.ipynb`, extended with the following improvements called out in the assessment and `captions.txt`:

| Feature | Source |
|---|---|
| **SQLite Q&A Knowledge Base** — Agent looks up pre-answered questions; unanswered ones are saved to DB and trigger a push notification | `captions.txt` / `4_lab4.ipynb` exercise |
| **Push Notifications** — Pushover alerts when a user leaves an email or when a new unanswerable question is logged | `4_lab4.ipynb` |
| **Evaluator Agent** — A second LLM (Gemini via OpenAI-compatible API) evaluates every response for quality; failed responses are automatically retried with feedback | `3_lab3.ipynb` |
| **Agent Loop from first principles** — Clean `while not done` loop drives all agentic behaviour, resolving tool calls before producing the final reply | `5_extra.ipynb` |

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from pydantic import BaseModel
import gradio as gr
import json
import os
import requests
import sqlite3

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

# Evaluator uses Gemini (via the OpenAI-compatible Google endpoint).
# Falls back to OpenAI gpt-4o-mini if GOOGLE_API_KEY is not set.
google_api_key = os.getenv("GOOGLE_API_KEY")
if google_api_key:
    evaluator_client = OpenAI(
        api_key=google_api_key,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )
    evaluator_model = "gemini-2.0-flash"
    print(f"Evaluator: Gemini ({evaluator_model})")
else:
    evaluator_client = openai
    evaluator_model = "gpt-4o-mini"
    print(f"Evaluator: OpenAI ({evaluator_model}) — set GOOGLE_API_KEY to use Gemini")

In [ ]:
# Resolve the 'me/' directory relative to this notebook.
# Jupyter sets CWD to the notebook's directory, so we walk up to find me/.
for candidate in ["../../me", "../me", "me"]:
    if os.path.isdir(candidate):
        me_dir = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the 'me' directory. "
        "Make sure you're running this notebook from inside community_contributions/jaymineh/"
    )

reader = PdfReader(os.path.join(me_dir, "linkedin.pdf"))
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open(os.path.join(me_dir, "summary.txt"), "r", encoding="utf-8") as f:
    summary = f.read()

name = "Jemine Mene-Ejegi"
print(f"Profile loaded for: {name}")
print(f"LinkedIn text: {len(linkedin):,} chars | Summary: {len(summary):,} chars")

In [ ]:
pushover_user  = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
PUSHOVER_URL   = "https://api.pushover.net/1/messages.json"

def push(message: str):
    """Send a push notification via Pushover. Prints to console if keys are not set."""
    print(f"[PUSH] {message}")
    if pushover_user and pushover_token:
        requests.post(PUSHOVER_URL, data={
            "user": pushover_user,
            "token": pushover_token,
            "message": message,
        })

if pushover_user and pushover_token:
    print(f"Pushover ready. User key starts with: {pushover_user[0]}")
else:
    print("Pushover keys not set — notifications will only appear in the console.")

## SQLite Q&A Knowledge Base

The agent gets two extra tools:
- **`lookup_qa`** — searches answered Q&A pairs so it can give richer, pre-verified answers.
- **`add_unanswered_question`** — saves a question the agent couldn't answer, stores it in the DB, and fires a push notification so the owner can supply the answer later.

This fulfils the exercise requirement: *"a database where the LM can add questions that require an answer ... it'll send you a push notification, and then you can come in and add the answers"* (`captions.txt`, lines 42–49).

In [ ]:
DB_PATH = "qa_database.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS qa_pairs (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            question    TEXT NOT NULL,
            answer      TEXT,
            answered    INTEGER DEFAULT 0,
            created_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    seed = [
        (
            "What are your strongest technical skills?",
            "My strongest skills include Kubernetes orchestration, Terraform IaC, "
            "multi-cloud architecture across AWS, GCP, and Azure, CI/CD pipelines with "
            "GitHub Actions, GitLab CI, Harness, and ArgoCD, Apache Kafka event streaming, "
            "HashiCorp Vault for secrets management, and LLM/AI infrastructure integration "
            "in private cloud environments.",
            1,
        ),
        (
            "Are you open to remote work?",
            "Yes, I'm open to fully remote roles as well as hybrid arrangements depending on location.",
            1,
        ),
        (
            "What industries have you worked in?",
            "I've worked primarily in fintech and security-driven environments, "
            "most recently as Senior Cloud/DevOps Engineer at Flutterwave.",
            1,
        ),
        (
            "What certifications do you hold?",
            "I hold CompTIA Security+, Microsoft Certified: Azure Fundamentals, "
            "Microsoft Certified: DevOps Engineer Expert, Technical Support Fundamentals, "
            "and DevOps Fundamentals certifications.",
            1,
        ),
        (
            "Have you worked with AI or LLMs?",
            "Yes — I've integrated LLMs into private cloud environments and built RAG-powered "
            "AI tooling that cut developer troubleshooting time by ~30% and increased inference "
            "accuracy by ~25%.",
            1,
        ),
        (
            "What is your preferred cloud platform?",
            "I'm genuinely multi-cloud — I've done deep production work on AWS, GCP, and Azure. "
            "Each has strengths; I choose based on the use case and existing organisational investment.",
            1,
        ),
        (
            "Where are you based?",
            "I'm based in Lagos State, Nigeria, originally from Jamaica. "
            "I'm open to relocating for the right opportunity.",
            1,
        ),
    ]
    cursor.executemany(
        "INSERT OR IGNORE INTO qa_pairs (question, answer, answered) VALUES (?, ?, ?)",
        seed,
    )
    conn.commit()
    conn.close()

init_db()
print(f"Q&A database ready at: {os.path.abspath(DB_PATH)}")

## Tool Functions

Four tools are available to the agent:

| Tool | Purpose |
|---|---|
| `record_user_details` | Save a user's email + push notify the owner |
| `record_unknown_question` | Log an unanswerable question + push notify |
| `lookup_qa` | Keyword-search the Q&A knowledge base |
| `add_unanswered_question` | Persist an important unanswered question to DB + push notify |

In [ ]:
def record_user_details(email: str, name: str = "Name not provided", notes: str = "not provided") -> dict:
    push(f"Interest from {name} | Email: {email} | Notes: {notes}")
    return {"recorded": "ok"}


def record_unknown_question(question: str) -> dict:
    push(f"Unanswerable question: {question}")
    return {"recorded": "ok"}


def lookup_qa(query: str) -> str:
    """Full-text keyword search across answered Q&A pairs."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "SELECT question, answer FROM qa_pairs "
        "WHERE answered = 1 AND (question LIKE ? OR answer LIKE ?)",
        (f"%{query}%", f"%{query}%"),
    )
    rows = cursor.fetchall()
    conn.close()
    if not rows:
        return f"No matching answered Q&A found for: '{query}'"
    result = f"Found {len(rows)} matching Q&A pair(s):\n\n"
    for q, a in rows:
        result += f"Q: {q}\nA: {a}\n\n"
    return result.strip()


def add_unanswered_question(question: str) -> dict:
    """Persist an unanswered question to the DB and notify the owner."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO qa_pairs (question, answered) VALUES (?, 0)",
        (question,),
    )
    conn.commit()
    conn.close()
    push(f"New unanswered question saved to DB: {question}")
    return {"recorded": "ok", "message": "Question saved — owner has been notified"}


print("Tool functions defined.")

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Record that a user is interested in getting in touch and has provided their email address.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The user's email address"},
            "name":  {"type": "string", "description": "The user's name, if provided"},
            "notes": {"type": "string", "description": "Any useful context about the conversation"},
        },
        "required": ["email"],
        "additionalProperties": False,
    },
}

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Record a question you couldn't answer based on your context. Always call this when you can't answer something.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

lookup_qa_json = {
    "name": "lookup_qa",
    "description": (
        "Search the Q&A knowledge base for pre-answered questions about Jemine. "
        "Always call this before saying you don't know something — there may be an answer already stored."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "The topic or keyword to search for in the knowledge base"},
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}

add_unanswered_question_json = {
    "name": "add_unanswered_question",
    "description": (
        "Save an important unanswered question to the database so the owner can provide an answer later. "
        "Use this for substantive questions that aren't in the knowledge base and that the owner should address."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question to save for the owner to answer"},
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json},
    {"type": "function", "function": lookup_qa_json},
    {"type": "function", "function": add_unanswered_question_json},
]
print(f"{len(tools)} tools registered: {[t['function']['name'] for t in tools]}")

In [ ]:
def handle_tool_calls(tool_calls: list) -> list:
    """Dispatch all tool calls and return a list of tool-role messages."""
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"[TOOL] {tool_name}({arguments})", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {"error": f"Tool '{tool_name}' not found"}
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id,
        })
    return results

In [ ]:
system_prompt = (
    f"You are acting as {name}. You are answering questions on {name}'s personal website, "
    f"particularly questions related to {name}'s career, background, skills, and experience. "
    f"Your responsibility is to represent {name} as faithfully and professionally as possible. "
    f"Be professional, warm, and engaging — as if speaking with a potential client or future employer.\n\n"
    f"Guidelines:\n"
    f"- Before saying you don't know something, ALWAYS call `lookup_qa` to check the knowledge base first.\n"
    f"- If you still can't answer after checking the knowledge base, call `record_unknown_question` to log it "
    f"AND call `add_unanswered_question` to save it to the database so the owner can answer it later.\n"
    f"- If a user seems interested in connecting, warmly ask for their email and record it with `record_user_details`.\n"
)
system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"Always stay in character as {name}."
print(f"System prompt ready ({len(system_prompt):,} chars).")

## Evaluator Agent

Borrowed from `3_lab3.ipynb`: a second LLM call evaluates every chatbot response for professionalism, accuracy, and helpfulness. If the response fails, the agent is given the feedback and asked to retry — implementing the **self-correction** agentic pattern.

In [ ]:
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


evaluator_system_prompt = (
    f"You are a quality evaluator for an AI chatbot that represents {name} on their personal website. "
    f"Your job is to decide whether the chatbot's latest response is acceptable. "
    f"Evaluate on: professionalism, factual accuracy relative to {name}'s known background, helpfulness, and tone. "
    f"The chatbot has been given this context:\n\n"
    f"## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
    f"Reply with whether the response is acceptable and provide clear, actionable feedback."
)


def evaluator_user_prompt(reply: str, message: str, history: list) -> str:
    prompt  = f"Conversation history:\n{json.dumps(history, indent=2)}\n\n"
    prompt += f"Latest user message:\n{message}\n\n"
    prompt += f"Agent's response:\n{reply}\n\n"
    prompt += "Is this response acceptable? Provide your evaluation."
    return prompt


def evaluate(reply: str, message: str, history: list) -> Evaluation:
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user",   "content": evaluator_user_prompt(reply, message, history)},
    ]
    try:
        response = evaluator_client.beta.chat.completions.parse(
            model=evaluator_model,
            messages=messages,
            response_format=Evaluation,
        )
    except Exception as e:
        if "429" in str(e) or "quota" in str(e).lower() or "RESOURCE_EXHAUSTED" in str(e):
            print(f"[EVAL] {evaluator_model} quota exceeded — falling back to OpenAI gpt-4o-mini")
            response = openai.beta.chat.completions.parse(
                model="gpt-4o-mini",
                messages=messages,
                response_format=Evaluation,
            )
        else:
            raise
    return response.choices[0].message.parsed


def rerun(reply: str, message: str, history: list, feedback: str) -> str:
    """Retry the agent response after evaluation failure, feeding back the rejection reason."""
    updated_prompt = (
        system_prompt
        + "\n\n## Quality Control: Previous Response Rejected\n"
        + "Your previous response was reviewed and rejected. Please improve it.\n"
        + f"\n### Your rejected response:\n{reply}\n"
        + f"\n### Reason for rejection:\n{feedback}\n\n"
        + "Please provide an improved response that addresses the feedback above."
    )
    messages = (
        [{"role": "system", "content": updated_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content


print(f"Evaluator ready (model: {evaluator_model}).")

## Agent Loop + Chat Function

The `chat()` function implements the **agent loop** pattern from `5_extra.ipynb`:

```
while not done:
    call LLM
    if tool_calls → execute tools, append results, loop again
    else           → done, get final reply
evaluate reply → if fails, rerun with feedback
```

In [ ]:
def chat(message: str, history: list) -> str:
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    # Agent loop — keep looping until there are no more tool calls
    done = False
    while not done:
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
        )
        finish_reason = response.choices[0].finish_reason

        if finish_reason == "tool_calls":
            tool_message = response.choices[0].message
            results = handle_tool_calls(tool_message.tool_calls)
            messages.append(tool_message)
            messages.extend(results)
        else:
            done = True

    reply = response.choices[0].message.content

    # Evaluate the final reply; retry once if it fails
    evaluation = evaluate(reply, message, history)
    if evaluation.is_acceptable:
        print(f"[EVAL] Passed — {evaluation.feedback[:80]}")
    else:
        print(f"[EVAL] Failed — {evaluation.feedback}")
        print("[EVAL] Retrying with feedback...")
        reply = rerun(reply, message, history, evaluation.feedback)

    return reply

## Validation

Run this cell to verify all components work before launching the UI.

In [ ]:
print("=" * 50)
print("Running validation checks...")
print("=" * 50)

# 1. record_user_details
result = record_user_details("test@example.com", "Test User", "validation run")
assert result == {"recorded": "ok"}, f"FAIL record_user_details: {result}"
print("PASS  record_user_details")

# 2. record_unknown_question
result = record_unknown_question("Do you hold a patent?")
assert result == {"recorded": "ok"}, f"FAIL record_unknown_question: {result}"
print("PASS  record_unknown_question")

# 3. lookup_qa — should find the skills entry
result = lookup_qa("Kubernetes")
assert "Kubernetes" in result, f"FAIL lookup_qa: {result}"
print(f"PASS  lookup_qa — snippet: {result[:80].strip()}...")

# 4. add_unanswered_question
result = add_unanswered_question("What is your biggest professional achievement?")
assert result["recorded"] == "ok", f"FAIL add_unanswered_question: {result}"
print("PASS  add_unanswered_question")

# 5. Evaluator — single LLM call to test structured output
print("\nTesting evaluator (makes one API call)...")
test_reply = (
    "My strongest skills are Kubernetes orchestration, Terraform IaC, and multi-cloud "
    "architecture across AWS, GCP, and Azure. I also have deep experience with CI/CD "
    "pipelines and LLM infrastructure integration."
)
eval_result = evaluate(test_reply, "What are your strongest technical skills?", [])
assert hasattr(eval_result, "is_acceptable"), "FAIL evaluate: missing is_acceptable"
assert hasattr(eval_result, "feedback"),      "FAIL evaluate: missing feedback"
print(f"PASS  evaluate — is_acceptable={eval_result.is_acceptable}")
print(f"       feedback: {eval_result.feedback[:120]}")

# 6. Verify DB has both answered and unanswered rows
conn = sqlite3.connect(DB_PATH)
answered_count   = conn.execute("SELECT COUNT(*) FROM qa_pairs WHERE answered = 1").fetchone()[0]
unanswered_count = conn.execute("SELECT COUNT(*) FROM qa_pairs WHERE answered = 0").fetchone()[0]
conn.close()
assert answered_count   >= 7, f"FAIL DB: expected >= 7 answered rows, got {answered_count}"
assert unanswered_count >= 1, f"FAIL DB: expected >= 1 unanswered row, got {unanswered_count}"
print(f"PASS  DB integrity — {answered_count} answered, {unanswered_count} unanswered")

print("\n" + "=" * 50)
print("All validations passed!")
print("=" * 50)

## Launch the Chatbot

In [ ]:
gr.ChatInterface(
    chat,
    type="messages",
    title=f"Chat with {name}",
    description=(
        f"Ask me anything about {name}'s career, skills, and experience. "
        "I'm an AI assistant representing them on this website."
    ),
).launch()